In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pdfplumber
import nltk
import spacy
from spacy.lang.en import English
from sentence_transformers import SentenceTransformer

from tqdm.auto import tqdm
import sqlite3
import re
from pathlib import Path
from zipfile import ZipFile


In [2]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

nlp = spacy.load("en_core_web_sm")

nlp_2 = English()
nlp_2.add_pipe("sentencizer")

[nltk_data] Downloading package stopwords to C:\Users\MY
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# import os

# # Path to your database
# db_path = "../Database/rag_documents.db"

# if os.path.exists(db_path):
#     os.unlink(db_path)  # or os.remove(db_path)
#     print(f"{db_path} deleted.")
# else:
#     print("Database file not found.")

In [3]:
DATAFILES_PATH = "../Data"
CONN = sqlite3.connect("../Database/rag_documents.db")

In [33]:
def create_tables():
    # Create a DB connector
    cursor = CONN.cursor()
    
    # Create unified table
    cursor.execute("DROP TABLE IF EXISTS documents")
    
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS pages (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        doc_name TEXT,
        character_count INTEGER,
        word_count INTEGER,
        sentence_count INTEGER,
        token_count INTEGER,
        text TEXT
    )
    """)
    
    CONN.commit()

In [ ]:
# Create a DB connector
cursor = CONN.cursor()

# Create unified table
cursor.execute("DROP TABLE IF EXISTS documents")

cursor.execute("""
CREATE TABLE IF NOT EXISTS pages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    doc_name TEXT,
    character_count INTEGER,
    word_count INTEGER,
    sentence_count INTEGER,
    token_count INTEGER,
    text TEXT
)
""")

CONN.commit()


In [39]:
def insert_page(doc_name, character_count, word_count, sentence_count, token_count, text):
    cursor = CONN.cursor()
    cursor.execute("""
        INSERT INTO pages (doc_name, character_count, word_count, sentence_count, token_count, text)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (doc_name, character_count, word_count, sentence_count, token_count, text))
    CONN.commit()

def extract_pages():
    cursor = CONN.cursor()
    cursor.execute("SELECT * FROM pages")
    rows = cursor.fetchall()
    df = pd.DataFrame(extract_pages(), columns=['id', 
                                                'doc_name', 
                                                'char_count', 
                                                'word_count',
                                                'sent_count', 
                                                'token_count',
                                                'text']).set_index('id')
    return df

def clear_pages():
    cursor = CONN.cursor()
    cursor.execute("DELETE FROM pages")
    CONN.commit()


In [ ]:
list(Path(DATAFILES_PATH).rglob("*"))

In [5]:
def get_file_list(data_path=DATAFILES_PATH):
    data_path = Path(data_path)
    dirs = []
    files = []

    for item in data_path.rglob("*"):
        if item.is_dir():
            dirs.append(item)
        else:
            files.append(item)

    all_files = [str(i) for i in files if not (Path(i).is_dir() or str(i).endswith(".zip"))]
    pdf_files = [str(i) for i in all_files if str(i).endswith(".pdf")]
    txt_files = [str(i) for i in all_files if str(i).endswith(".txt")]
    csv_files = [str(i) for i in all_files if str(i).endswith(".csv")]
    
    return all_files, pdf_files, txt_files, csv_files

In [6]:
all_files, pdf_files, txt_files, csv_files = get_file_list()


In [ ]:
def clean_txt_file(txt_file):
    doc_name = Path(txt_file).stem
    doc_type = Path(txt_file).suffix

    with open(txt_file, 'r', encoding='utf-8', errors='ignore') as file:
        text = file.read()
    
    cleaned_text = text.capitalize()
    
    insert_page(doc_name,
            len(cleaned_text),
            len(cleaned_text.split(" ")),
            len(list(nlp_2(cleaned_text).sents)),
            len(nlp(cleaned_text)),
            cleaned_text)

    return None

In [ ]:
def clean_csv_file(csv_file, batch_size=50):
    doc_name = Path(csv_file).stem
    doc_type = Path(csv_file).suffix
    
    df = pd.read_csv(csv_file)

    df.dropna(inplace=True)  # drop rows with any NaN values
    df = df.select_dtypes(include=[object])  # select only string columns

    def clean_text(row):
        new_list = []
        for col in row.index:  # iterate through columns
            new_list.append(f"{col.capitalize()}: {row[col]}")  
        return ". ".join(new_list)
    
    for i in range(0, len(df), batch_size):
        batch_df = df.iloc[i:i+batch_size]
        cleaned_text = batch_df.apply(clean_text, axis=1).to_list()
        cleaned_text = ' '.join(cleaned_text)
    
        insert_page(doc_name,
                    len(cleaned_text),
                    len(cleaned_text.split(" ")),
                    len(list(nlp_2(cleaned_text).sents)),
                    len(nlp(cleaned_text)),
                    cleaned_text)
    
    return None

In [12]:
def classify_page(text):
    doc = nlp(text)
    lowered = text.lower()
    num_lines = len(text.splitlines())

    # Heuristic 1: Sentence length
    sents = list(doc.sents)
    avg_len = sum(len(sent.text) for sent in sents) / max(1, len(sents))

    # Heuristic 2: POS distribution
    pos_counts = doc.count_by(spacy.attrs.POS)
    num_verbs = pos_counts.get(doc.vocab.strings["VERB"], 0)
    num_nouns = pos_counts.get(doc.vocab.strings["NOUN"], 0)
    num_propns = pos_counts.get(doc.vocab.strings["PROPN"], 0)

    # Heuristic 3: Numbers/digits
    num_digits = sum(c.isdigit() for c in text)
    digit_ratio = num_digits / max(1, len(text))

    # ------------------ RULES ------------------ #

    # References / Bibliography → many proper nouns + numbers + citations
    if ("references" in lowered or "bibliography" in lowered) or \
       (num_propns > 15 and num_verbs < 5 and digit_ratio > 0.05):
        return "reference"

    # TOC → many short lines, dotted leaders, section/chapter keywords
    if ("table of contents" in lowered or "contents" in lowered) or \
       (re.search(r"\.{5,}\s*\d+", text) and num_verbs < 5 and num_lines > 5):
        return "toc"

    # Acknowledgement → gratitude words + relatively short text
    if any(word in lowered for word in ["thank", "thankful","grateful", "acknowledgements" ,"acknowledgement", "debt of gratitude", "appreciation", "appreciations"]):
        if len(text) < 1000:  # avoid false positives in main body
            return "acknowledgement"

    # If nothing matched → assume content
    return "content"

In [ ]:
def clean_pdf_file(pdf_file, first_page=None, last_page=None):
    doc_name = Path(pdf_file).stem
    
    with pdfplumber.open(pdf_file) as pdf:
        if first_page or last_page:
            pages = pdf.pages[first_page:last_page]
            for i, page in enumerate(pages, start=first_page or 0):
                if text := page.extract_text(x_tolerance=1, y_tolerance=1).replace('\n', ' '):
                    if text.strip():  # ignore empty pages
                        insert_page(doc_name,
                                    len(text),
                                    len(text.split(" ")),
                                    len(list(nlp_2(text).sents)),
                                    len(nlp(text)),
                                    text)
                        
        else:
            for i, page in enumerate(pdf.pages):
                if text := page.extract_text(x_tolerance=1, y_tolerance=1).replace('\n', ' '):
                    if text.strip():  # ignore empty pages
                        if classify_page(text) not in ["reference", "toc", "acknowledgement"]:
                            insert_page(doc_name,
                                        len(text),
                                        len(text.split(" ")),
                                        len(list(nlp_2(text).sents)),
                                        len(nlp(text)),
                                        text)

    
    return None

In [ ]:
for file in tqdm(enumerate(txt_files)):
    clean_txt_file(file)

for file in tqdm(enumerate(csv_files)):
    clean_csv_file(file)

for file in tqdm(enumerate(pdf_files)):
    clean_pdf_file(file)

In [ ]:
pages = extract_pages()